# Reinforcement Learning Practice Notebook

- Id: 22302421
- Full name: Nguyễn Phúc Minh Châu

This notebook covers three topics:
- Q-Learning
- Approximate Q-Learning
- Policy Search

The environment is a 3x3 Grid World with obstacles and a terminal state.

## Environment Description

- Grid size: 3x3 (states 0 to 8)
- Start state: 0
- Terminal state: 8
- Obstacle state: 4 (cannot enter)
- Actions: 0=up, 1=down, 2=left, 3=right
- Reward: -1 per step, +10 at terminal state

In [1]:
import random
import math

# Định nghĩa các tham số môi trường
num_states = 9                 # Tổng số trạng thái
num_actions = 4                # Số hành động
terminal_state = 8             # Trạng thái kết thúc
obstacle_state = 4             # Trạng thái chướng ngại

def step(state, action):
    # Nếu đã ở trạng thái kết thúc thì không đi tiếp
    if state == terminal_state:
        return state, 0

    row, col = divmod(state, 3)   # Chuyển state thành tọa độ

    # Xác định vị trí mới dựa trên hành động
    if action == 0: row -= 1       # Đi lên
    elif action == 1: row += 1     # Đi xuống
    elif action == 2: col -= 1     # Sang trái
    elif action == 3: col += 1     # Sang phải

    # Kiểm tra biên
    if row < 0 or row > 2 or col < 0 or col > 2:
        return state, -1

    next_state = row * 3 + col

    # Kiểm tra chướng ngại
    if next_state == obstacle_state:
        return state, -1

    # Kiểm tra trạng thái kết thúc
    if next_state == terminal_state:
        return next_state, 10

    return next_state, -1

## Part 1: Q-Learning

In [2]:
# Khởi tạo bảng Q
Q = [[0.0 for _ in range(num_actions)] for _ in range(num_states)]

alpha = 0.1    # Tốc độ học
gamma = 0.9    # Hệ số chiết khấu
epsilon = 0.1  # Xác suất exploration

for episode in range(500):
    state = 0  # Bắt đầu từ trạng thái 0

    while state != terminal_state:
        # Chọn hành động theo epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, num_actions - 1)
        else:
            action = max(range(num_actions), key=lambda a: Q[state][a])

        next_state, reward = step(state, action)

        # Cập nhật Q-value
        best_next = max(Q[next_state])
        Q[state][action] += alpha * (reward + gamma * best_next - Q[state][action])

        state = next_state

### Applying the Learned Policy

In [9]:
policy = []  # Danh sách lưu policy học được

for s in range(num_states):
    # Với mỗi trạng thái, chọn hành động có Q-value lớn nhất
    best_action = Q[s].index(max(Q[s]))
    policy.append(best_action)

state = 0  # Trạng thái bắt đầu
trajectory = [state]  # Lưu lại đường đi của agent
actions = [] # Lưu chuỗi các hành động mà agent thực hiện

while state != terminal_state:
    # Chọn hành động theo policy đã học
    action = policy[state]

    # Thực hiện hành động
    state, _ = step(state, action)
    trajectory.append(state)
    actions.append(action)

print("Agent trajectory:", trajectory)  # In đường đi của agent
print("Sequence of actions:", actions) # In chuỗi các hành động của agent

Agent trajectory: [0, 3, 6, 7, 8]
Sequence of actions: [1, 1, 3, 3]


## Part 2: Approximate Q-Learning

In [3]:
# Khởi tạo vector trọng số
num_features = num_states * num_actions
weights = [0.0 for _ in range(num_features)]

def features(state, action):
    # Vector đặc trưng one-hot cho (state, action)
    f = [0.0 for _ in range(num_features)]
    index = state * num_actions + action
    f[index] = 1.0
    return f

def q_value(state, action):
    # Tính Q(s,a) = w · f(s,a)
    f = features(state, action)
    return sum(w * x for w, x in zip(weights, f))

for episode in range(500):
    state = 0

    while state != terminal_state:
        # Chọn hành động epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, num_actions - 1)
        else:
            action = max(range(num_actions), key=lambda a: q_value(state, a))

        next_state, reward = step(state, action)

        # Tính TD error
        target = reward + gamma * max(q_value(next_state, a) for a in range(num_actions))
        delta = target - q_value(state, action)

        # Cập nhật trọng số
        f = features(state, action)
        for i in range(num_features):
            weights[i] += alpha * delta * f[i]

        state = next_state

### Applying the Learned Policy

In [11]:
state = 0
trajectory = [state]  # Lưu lại đường đi của agent
actions = [] # Lưu chuỗi các hành động mà agent thực hiện

while state != terminal_state:
    # Chọn hành động có Q lớn nhất
    action = max(range(num_actions), key=lambda a: q_value(state, a))
    
    # Thực hiện hành động
    state, _ = step(state, action)
    trajectory.append(state)
    actions.append(action)

print("Agent trajectory:", trajectory)  # In đường đi của agent
print("Sequence of actions:", actions) # In chuỗi các hành động của agent

Agent trajectory: [0, 1, 2, 5, 8]
Sequence of actions: [3, 3, 1, 1]


## Part 3: Policy Search

In [8]:
# Khởi tạo bảng preference
preferences = [[0.0 for _ in range(num_actions)] for _ in range(num_states)]

def softmax(action_preferences):
    # action_preferences nên là một list các con số, ví dụ: [1.2, 0.5, 2.0]
    exp_vals = [math.exp(p) for p in action_preferences]
    total = sum(exp_vals)
    return [v / total for v in exp_vals]

def policy(state):
    # Truy cập preferences của state trước khi đưa vào softmax
    return softmax(preferences[state])

def choose_action(state):
    # Lấy phân phối xác suất của các hành động
    probs = policy(state)
    # Lấy một số ngẫu nhiên trong khoảng [0, 1)
    r = random.random()
    # Duyệt qua các hành động để chọn theo xác suất
    cumulative = 0.0                 # Khởi tạo tổng xác suất tích lũy
    for action, prob in enumerate(probs):  # Duyệt từng hành động và xác suất tương ứng
        cumulative += prob           # Cộng dồn xác suất
        if r < cumulative:           # Nếu số ngẫu nhiên r rơi vào khoảng này
            return action            # Chọn hành động hiện tại
    return num_actions - 1           # Trường hợp biên do sai số số học, chọn hành động cuối

In [9]:
# Vòng lặp huấn luyện policy
for episode in range(50):
    state = 0
    trajectory = []
    rewards = []
    
    # Sinh một episode theo policy hiện tại
    while state != terminal_state:
        action = choose_action(state)
        next_state, reward = step(state, action)
        trajectory.append((state, action))
        rewards.append(reward)
        state = next_state
    
    # Tổng phần thưởng của episode
    G = sum(rewards)
    
    # Cập nhật policy parameters
    for (state, action) in trajectory:
        probs = policy(state)
        for a in range(num_actions):
            if a == action:
                # Tăng xác suất của hành động đã chọn
                preferences[state][a] += alpha * G * (1 - probs[a])
            else:
                # Giảm xác suất của các hành động khác
                preferences[state][a] -= alpha * G * probs[a]

### Applying the Learned Policy

In [10]:
# Áp dụng policy đã học
state = 0
terminal_state = terminal_state  # giả sử đã được định nghĩa trước
max_steps = 100                  # số bước tối đa để tránh lặp vô hạn

trajectory = [state]  # Lưu lại đường đi của agent
actions = []          # Lưu chuỗi các hành động mà agent thực hiện

step_count = 0

while state != terminal_state and step_count < max_steps:
    # Lấy xác suất hành động từ policy
    probs = policy(state)
    
    # Chọn hành động có xác suất cao nhất (greedy)
    action = probs.index(max(probs))
    
    # Thực hiện hành động và chuyển sang trạng thái mới
    state, _ = step(state, action)
    
    trajectory.append(state)
    actions.append(action)
    
    # Tăng bộ đếm số bước
    step_count += 1

print("Agent trajectory:", trajectory)   # In đường đi của agent
print("Sequence of actions:", actions)   # In chuỗi các hành động của agent

# Thông báo lý do kết thúc
if state == terminal_state:
    print("Episode ended: reached terminal state.")
else:
    print("Episode ended: reached maximum number of steps.")

Agent trajectory: [0, 1, 2, 5, 8]
Sequence of actions: [3, 3, 1, 1]
Episode ended: reached terminal state.
